# 字符串与 Unicode

学习目标：能处理字符串并区分 UTF-16 码元、Unicode 码点和显示字符，正确选择 URI 编码方式。

前置知识：原始值不可变性、索引概念、算术与比较、显式字符串转换。

适用版本：ECMAScript 2025（ECMA-262 第 16 版）、Node.js 24.11.0；.mjs 文件按 ES 模块运行并采用严格模式。console 是宿主输出 API。

环境准备：[环境配置与运行](README.md)。

工作目录：content/编程语言/javascript。

配套脚本：位于 scripts/05-strings-and-unicode/。

1. [main.mjs](scripts/05-strings-and-unicode/main.mjs)：按正文顺序运行全部正常示例。
2. [string-write.mjs](scripts/05-strings-and-unicode/string-write.mjs)：严格模式下给字符串索引赋值失败，不能用它修改原始字符串。
3. [uri-invalid.mjs](scripts/05-strings-and-unicode/uri-invalid.mjs)：孤立代理码元无法按这里的 URI 编码规则转换为 UTF-8。

Step 1：从项目根目录进入本章工作目录。

```bash
cd content/编程语言/javascript
```

Step 2：运行全部正常示例，按各片段中的输出注释核对。

```bash
node scripts/05-strings-and-unicode/main.mjs
```

下文正常片段依次对应 main.mjs 中的代码；每段给出自身输入与定义。错误文件仅在相应小节单独运行。

## 1 不可变字符串、索引与切片

字符串是不可变的 UTF-16 码元序列；变量可以改为保存新字符串，但不会修改旧字符串。索引从 0 开始，length 计数码元，超界方括号访问得到 undefined，charAt() 超界则返回空字符串。

slice(start, end) 返回从 start 到 end 之前的部分，两个参数都是码元位置；负数从末尾计算，省略 end 表示直到结尾。substring() 将负数当成 0，并在起点大于终点时交换二者，不能把两种方法的边界混为一谈。

```javascript
const text = "JavaScript";
console.log(text.length, text[0], text.at(-1), text[99]);
console.log(text.charAt(99) === "");
console.log(text.slice(0, 4), text.slice(-6), text.slice(4, 0) === "");
console.log(text.substring(4, 0), text.substring(-2, 4));
const edited = "Type" + text.slice(4);
console.log(text, edited);
// 输出依次为：
// 10 J t undefined
// true
// Java Script true
// Java Java
// JavaScript TypeScript
```

## 2 查找、清理与替换

includes() 判断是否包含，indexOf() 返回第一次出现的码元索引或 -1；位置 0 是有效命中，不能把 indexOf() 直接当布尔条件。startsWith()、endsWith() 判断前后缀，默认区分大小写。

trim() 删除两端空白，不清除中间空白。replace() 使用字符串模式时只替换第一次；replaceAll() 替换全部匹配的字符串，均返回新字符串。替换字符串中的 $& 等序列有特殊含义，需要字面美元符号时按替换规则处理；本节使用不含这些序列的替换文本。正则模式在正则表达式章节展开。

```javascript
const route = "  js/js/notes  ";
const clean = route.trim();
console.log(clean, clean.includes("js"), clean.indexOf("js"), clean.indexOf("ts"));
console.log(clean.startsWith("js/"), clean.endsWith("notes"));
console.log(clean.replace("js", "ts"), clean.replaceAll("js", "ts"));
console.log("ab".toUpperCase(), "AB".toLowerCase(), "7".padStart(3, "0"));
// 输出依次为：
// js/js/notes true 0 -1
// true true
// ts/js/notes ts/ts/notes
// AB ab 007
```

## 3 模板、转义与 String.raw

模板字面量使用反引号定界，${expression} 将表达式的结果插入文本，其中 expression 是要计算的 JavaScript 表达式。模板可以跨行；单引号和双引号字符串不能直接包含普通换行，通常用转义序列。

\n、\t、\\ 分别表示换行、制表符和反斜杠；\u{1F680} 表示指定 Unicode 码点，\u0041 表示十六进制码元。String.raw 放在模板前时保留模板文本里的反斜杠写法，但插值表达式仍先正常求值；它不会再次“还原”插入值中的转义。String.raw 是标签函数，调用机制在函数章节展开。

```javascript
const name = "Ada";
const tasks = 2;
console.log(`${name} 完成 ${tasks + 1} 项`);
console.log("A\nB".length, "A\tB".length, "\\".length);
console.log("\u0041", "\u{1F680}");
const rawText = String.raw`A\nB`;
const interpolated = String.raw`A${"\n"}B`;
console.log(rawText, rawText.length, interpolated.length);
// 输出依次为：
// Ada 完成 3 项
// 3 3 1
// A 🚀
// A\nB 4 3
```

## 4 UTF-16 码元与 Unicode 码点

码点（code point）是 Unicode 中的编号；码元（code unit）是编码中的存储单位。JavaScript 字符串的一个码元占 16 位。基本多文种平面之外的码点使用一对代理码元（surrogate pair）表示，因此一个符号可能使 length 增加 2。

charCodeAt() 读取单个码元，codePointAt() 从给定码元索引读取码点；若从代理对的后半位置读取，并不会自动回到前半位置。String.fromCodePoint() 从码点构造字符串。slice() 也按码元切片，可能切断代理对。

for...of 逐个迭代码点，写法中的 unit 是每轮得到的字符串值；循环语法下一章展开。码点仍不等于用户看到的一个字符，例如字母与组合音标可由多个码点组成；按字素簇分段在国际化章节使用 Intl.Segmenter。

```javascript
const label = "A🚀B";
console.log(label.length, label.charCodeAt(1).toString(16), label.codePointAt(1).toString(16));
console.log(label.codePointAt(2).toString(16), String.fromCodePoint(0x1F680));
console.log(label.slice(1, 3), label.slice(1, 2).isWellFormed());
let codePoints = 0;
for (const unit of label) {
  codePoints += 1;
}
console.log(codePoints);
console.log("é".length);
// 输出依次为：
// 4 d83d 1f680
// de80 🚀
// 🚀 false
// 3
// 2
```

## 5 规范化与不完整 Unicode

看起来相同的文本可能由不同码点序列组成，严格相等不会自动规范化。normalize() 默认采用 NFC，将规范等价序列转换到统一的组合形式；NFD 使用分解形式。NFKC 与 NFKD 还做兼容转换，可能折叠有意义的排版差异，应根据字段语义选择，不能把“规范化”理解为所有视觉相似文本都相等。

JavaScript 字符串允许孤立代理码元，isWellFormed() 检查是否存在这种不完整的 UTF-16 序列；toWellFormed() 将孤立代理码元替换为 U+FFFD。替换会丢弃原来的无效片段，应与拒绝输入的策略区分。

```javascript
const composed = "\u00E9";
const decomposed = "e\u0301";
console.log(composed === decomposed, composed === decomposed.normalize("NFC"));
console.log(composed.normalize("NFD").length);
console.log("①".normalize("NFC"), "①".normalize("NFKC"));
const broken = "\uD800";
console.log(broken.isWellFormed(), broken.toWellFormed().codePointAt(0).toString(16));
// 输出依次为：
// false true
// 2
// ① 1
// false fffd
```

## 6 URI 编码与组件编码

encodeURI() 适合编码已经具有 URI 结构的文本，会保留冒号、斜杠、问号、&、# 等结构字符。encodeURIComponent() 用于单个组件，会编码这些可能改变结构的字符；例如搜索词中的 & 不应被当作下一个查询参数的分隔符。

两者均按 UTF-8 字节进行百分号编码，不是加密，也不检查地址是否安全。它们不会将空格改为 +，表单编码是另一套规则。decodeURI() 保留对 URI 结构有意义的转义，decodeURIComponent() 则可解开这些组件字符；应配对使用并避免对已经编码的数据重复编码。

孤立代理码元导致编码函数抛出 URIError，不完整的百分号序列也会使解码失败。下面只构造本地字符串，不访问外网。

```javascript
const address = "https://example.test/search?q=学习";
console.log(encodeURI(address));
const term = "a&b c";
const component = encodeURIComponent(term);
console.log(component, decodeURIComponent(component));
console.log(decodeURI("%2F"), decodeURIComponent("%2F"));
console.log("/search?q=" + component);
// 输出依次为：
// https://example.test/search?q=%E5%AD%A6%E4%B9%A0
// a%26b%20c a&b c
// %2F /
// /search?q=a%26b%20c
```

## 7 独立观察错误边界

以下文件分别启动新进程；预期退出码为 1。先根据代码判断错误原因，再运行相应命令，核对错误名称及对应位置。错误消息全文由宿主决定。

严格模式下给字符串索引赋值失败，不能用它修改原始字符串。

```javascript
const text = "abc";
text[0] = "A";
// 预期错误：TypeError；Cannot assign to read only property '0'
```

Step 1：独立运行 scripts/05-strings-and-unicode/string-write.mjs。

```bash
node scripts/05-strings-and-unicode/string-write.mjs
```

孤立代理码元无法按这里的 URI 编码规则转换为 UTF-8。

```javascript
console.log(encodeURIComponent("\uD800"));
// 预期错误：URIError；URI malformed
```

Step 2：独立运行 scripts/05-strings-and-unicode/uri-invalid.mjs。

```bash
node scripts/05-strings-and-unicode/uri-invalid.mjs
```

## 本章小结

- 字符串操作产生新值；索引、length 和 slice 使用 UTF-16 码元。
- 码元、码点、字素簇属于不同层次，逐码点遍历也不能数尽所有显示字符。
- 规范化需要明确语义；URI 整体编码和组件编码保护的结构不同。

## 练习

1. 从 "A🚀B" 提取完整火箭并验证 length 为 2、codePointAt(0) 为 0x1F680；比较切掉一个码元后 isWellFormed() 的结果。
2. 将 "e\u0301" 的转义写法作为源码字符串与 "é" 比较，再统一到 NFC；核对规范化前不相等、之后相等，原字符串不变。
3. 将 "x/y?z=1&k=2" 当作一个搜索词编码并往返解码；核对斜杠、问号、等号、& 都未被误当作 URL 结构。
4. 用 replace() 和 replaceAll() 处理三次出现的同一单词；核对只替换一次与全部替换的差异，并保留原文本。

## 参考与引用来源

- TC39（tc39.es）：[§6.1.4 String 类型](https://tc39.es/ecma262/2025/multipage/ecmascript-data-types-and-values.html#sec-ecmascript-language-types-string-type)、[§12.9.4 字符串与转义](https://tc39.es/ecma262/2025/multipage/ecmascript-language-lexical-grammar.html#sec-literals-string-literals)、[§13.2.8 模板字面量](https://tc39.es/ecma262/2025/multipage/ecmascript-language-expressions.html#sec-template-literals)、[§22.1 String：索引、切片、查找、替换、String.raw、规范化和迭代](https://tc39.es/ecma262/2025/multipage/text-processing.html#sec-string-objects)、[§19.2.6 URI 编码与解码](https://tc39.es/ecma262/2025/multipage/global-object.html#sec-uri-handling-functions)：ECMAScript 2025 字符串规则；normalize() 的 Unicode 规范化定位见 §22.1.3.15。